# Day 1 - Setting up the workshop

**Big idea:** before building anything with AI, make the project *reproducible* (works the same on every machine) and *safe* (secrets never leak).

## 1. `pyproject.toml` vs `uv.lock` - shopping list vs receipt

- **`pyproject.toml`** = the shopping list. "I want `fastapi`, version 0.141 or newer."
- **`uv.lock`** = the receipt. "You got *exactly* fastapi 0.141.1, plus these exact 50+ other packages it pulled in."

Without the lock file, "0.141 or newer" can mean a different version next month - and a surprise update can break your code. The lock file freezes the whole tree, so `uv sync` rebuilds the *same* environment for everyone.

Let's look at both files for real:

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

import tomllib

pyproject = tomllib.loads((ROOT / "pyproject.toml").read_text())
print("The shopping list (pyproject.toml) says:")
for dep in pyproject["project"]["dependencies"]:
    print("   ", dep)

The shopping list (pyproject.toml) says:
    anthropic>=1.6.0
    asyncpg>=0.31.0
    dotenv>=0.9.9
    fastapi>=0.141.1
    google-genai>=2.24.0
    httpx>=0.28.1
    openai>=3.15.0
    python-dotenv>=1.2.3
    requests>=2.34.2
    sse-starlette>=3.4.11
    tiktoken>=0.14.0
    uvicorn[standard]>=0.53.0


In [2]:
lock = tomllib.loads((ROOT / "uv.lock").read_text())
pinned = {pkg["name"]: pkg["version"] for pkg in lock["package"] if "version" in pkg}

print(f"The receipt (uv.lock) pins {len(pinned)} packages exactly. A few of them:")
for name in ["fastapi", "tiktoken", "httpx", "starlette", "anyio"]:
    print(f"    {name:<10} == {pinned.get(name)}")

The receipt (uv.lock) pins 84 packages exactly. A few of them:
    fastapi    == 0.141.1
    tiktoken   == 0.14.0
    httpx      == 0.28.1
    starlette  == 1.6.0
    anyio      == 4.15.1


Notice `starlette` and `anyio` - I never asked for those, `fastapi` needs them. These are **transitive dependencies**, and they're exactly the ones most likely to change under you without a lock file.

**Commands to remember**
- `uv add <pkg>` - adds it and updates both files.
- `uv sync` - installs exactly what `uv.lock` says.
- `uv run <cmd>` - runs a command inside the project's environment.

## 2. Secrets live in `.env`, and `.env` never goes into git

API keys go in a `.env` file. Code reads them with `python-dotenv` + `os.getenv`, and `.gitignore` keeps the file out of git. `.env.example` (same keys, no real values) *is* committed, so others know what to set.

Rule: **never print a real key - not even part of it.** Just confirm it loaded:

In [3]:
import os
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

def status(value):
    return f"set ({len(value)} chars)" if value else "(not set)"

for key in ["GEMINI_API_KEY", "OPEN_ROUTER_API_KEY", "OLLAMA_HOST", "DATABASE_URL"]:
    print(f"{key:<20} {status(os.getenv(key))}")

GEMINI_API_KEY       set (53 chars)
OPEN_ROUTER_API_KEY  set (73 chars)
OLLAMA_HOST          set (22 chars)
DATABASE_URL         set (45 chars)


In [4]:
gitignore = (ROOT / ".gitignore").read_text().splitlines()
print(".env is ignored by git:", ".env" in gitignore)

.env is ignored by git: True


### Why deleting a pushed `.env` doesn't save you

Git is a **photo album**, not a whiteboard. Every commit is a photo that stays forever. If you commit `.env` and delete it in the next commit, the old photo still has your key in it - and anyone who cloned or scraped the repo already has a copy.

The fix, in this order:
1. **Rotate the key first** (make the leaked one useless). This is the only step that actually protects you.
2. Then scrub history (`git filter-repo` or BFG) and force-push.
3. Treat the old key as compromised forever.

## 3. My goals (by 24 Jan 2027)

1. A production-ready AI backend: FastAPI + Postgres + Redis + auth + background jobs + Docker, deployed publicly.
2. A multi-tenant RAG API with citations, tenant isolation, and an automated eval suite in CI.
3. A portfolio of 3 deployed AI projects I can explain confidently in an interview.

Days 2-6 are all building blocks toward #1: calling models (D2), streaming (D3), cost (D5), observability (D6).

## Recap

- `pyproject.toml` = what you want. `uv.lock` = exactly what you got. Commit both.
- Secrets in `.env`, `.env` in `.gitignore`, `.env.example` committed.
- A pushed secret is burned: rotate first, clean history second.